# Plotting Clusters
## 1. Cancer

In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo
import pandas as pd
import medicc
%matplotlib inline
%reload_ext autoreload
%autoreload 2
results_dir = "/home/ganesank/project/phytclust/results"  # change

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

def plot_cn_heatmap(
    input_df,
    cluster_df,
    final_tree=None,
    y_posns=None,
    cmax=None,
    total_copy_numbers=False,
    alleles=["cn_a", "cn_b"],
    tree_width_ratio=1,
    cbar_width_ratio=0.05,
    figsize=(18, 8),
    tree_line_width=0.5,
    tree_marker_size=0,
    show_internal_nodes=False,
    title="",
    tree_label_colors=None,
    tree_label_func=None,
    cmap="coolwarm",
    cluster_cmap="tab20b",
    outgroup="diploid",
    ignore_segment_lengths=False,
    save_path=None,
):

    input_df = input_df[alleles].copy()
    cluster_df = cluster_df.copy()

    if not isinstance(alleles, (list, tuple)):
        alleles = [alleles]
    nr_alleles = len(alleles)

    cmax = cmax or np.max(input_df[alleles].values.astype(int))

    if final_tree is None:
        fig, axs = plt.subplots(
            figsize=figsize,
            ncols=1 + nr_alleles + 1,
            sharey=False,
            gridspec_kw={"width_ratios": [0.1] + nr_alleles * [1] + [cbar_width_ratio]},
        )

        cur_sample_labels = input_df.index.get_level_values("sample_id").unique()
        if not show_internal_nodes:
            print('No tree provided, so "show_internal_nodes=False" is ignored')
        y_posns = y_posns or {s: i for i, s in enumerate(cur_sample_labels)}

        clust_ax = axs[0]
        cn_axes = axs[1:-1]
        cax = axs[-1]

        cn_axes[0].set_title(
            title,
            x=0,
            y=1,
            ha="left",
            va="bottom",
            pad=20,
            fontweight="bold",
            fontsize=16,
            zorder=10,
        )
    else:
        fig, axs = plt.subplots(
            figsize=figsize,
            ncols=2 + nr_alleles + 1,
            sharey=False,
            gridspec_kw={
                "width_ratios": [tree_width_ratio]
                + [0.2]
                + nr_alleles * [1]
                + [cbar_width_ratio]
            },
        )
        tree_ax = axs[0]  # Tree
        clust_ax = axs[1]  # Clusters
        cn_axes = axs[2:-1]  # Copy number
        cax = axs[-1]  # Colorbar

        if show_internal_nodes:
            cur_sample_labels = np.array(
                [x.name for x in final_tree.find_clades() if x.name is not None]
            )
        else:
            cur_sample_labels = np.array([x.name for x in final_tree.get_terminals()])

        y_posns = {
            k.name: v
            for k, v in _get_y_positions(
                final_tree, adjust=show_internal_nodes, outgroup=outgroup
            ).items()
        }

        _ = plot_tree(
            final_tree,
            ax=tree_ax,
            outgroup=outgroup,
            label_func=tree_label_func if tree_label_func is not None else lambda x: "",
            hide_internal_nodes=(not show_internal_nodes),
            show_branch_lengths=False,
            show_events=False,
            line_width=tree_line_width,
            marker_size=tree_marker_size,
            title=title,
            label_colors=tree_label_colors,
        )
        tree_ax.set_axis_off()
        fig.set_constrained_layout_pads(w_pad=0, h_pad=0, hspace=0.0, wspace=50)

    # Calculate gaps
    gaps = (
        input_df.loc[cur_sample_labels[0]].eval("start")
        - np.roll(input_df.loc[cur_sample_labels[0]].eval("end"), 1)
    ).values
    total_gaps = gaps[gaps > 0].sum()
    if total_gaps > 1e8:
        print(
            f"Total of {total_gaps:.1e} bp gaps in the segmentation. These missing "
            "segments are not reflected in the plot!"
        )

    # Sort labels
    ind = [y_posns.get(x, -1) for x in cur_sample_labels]
    cur_sample_labels = cur_sample_labels[np.argsort(ind)]

    # Color normalization
    vcenter = 2 if total_copy_numbers else 1
    color_norm = mcolors.TwoSlopeNorm(
        vmin=0, vcenter=vcenter, vmax=cmax if cmax > vcenter else vcenter + 1
    )

    # Chromosome ends
    chr_ends = input_df.loc[cur_sample_labels[0]].copy()
    chr_ends["end_pos"] = np.cumsum([1] * len(chr_ends))
    chr_ends = (
        chr_ends.reset_index().groupby("chrom").max()["end_pos"].dropna().astype(int)
    )

    # x positions
    if ignore_segment_lengths:
        x_pos = np.arange(len(input_df.loc[cur_sample_labels].unstack("sample_id")) + 1)
    else:
        x_pos = np.append(
            [0],
            np.cumsum(
                input_df.loc[cur_sample_labels]
                .unstack("sample_id")
                .loc[:, alleles[0]]
                .loc[:, cur_sample_labels]
                .eval("end+1-start")
                .values
            ),
        )
    y_pos = np.arange(len(cur_sample_labels) + 1) + 0.5

    # Prepare and plot the cluster heatmap
    cmax_clust = np.max(cluster_df.values.astype(int))
    cluster_labels = cluster_df.index.get_level_values("leaf_name").unique()
    ind_clust = [y_posns.get(x, -1) for x in cluster_labels]
    cluster_labels = cluster_labels[np.argsort(ind_clust)]
    color_norm_clust = mcolors.Normalize(vmin=1, vmax=cmax_clust)
    fixed_x_range = (10000, 50000)
    solution_ends = cluster_df.loc[cluster_labels[0]].copy()
    solution_ends["end_pos"] = np.cumsum([1] * len(solution_ends))
    solution_ends = (
        solution_ends.reset_index()
        .groupby("comparison_IDs")
        .max()["end_pos"]
        .dropna()
        .astype(int)
    )
    x_pos_clust = np.linspace(
        fixed_x_range[0],
        fixed_x_range[1],
        len(cluster_df.loc[cluster_labels].astype(int).unstack("leaf_name")) + 1,
    )
    y_pos_clust = np.arange(len(cluster_labels) + 1) + 0.5
    cluster_cmap = plt.get_cmap(cluster_cmap)
    if cluster_cmap.N < cmax_clust:
        cluster_cmap = ListedColormap(
            np.vstack(
                [cluster_cmap(np.linspace(0, 1, cluster_cmap.N))]
                * (cmax_clust // cluster_cmap.N + 1)
            )
        )
    im_new = clust_ax.pcolormesh(
        x_pos_clust,
        y_pos_clust,
        cluster_df.loc[cluster_labels]
        .astype(int)
        .unstack("leaf_name")
        .loc[:, "cluster_ID"]
        .loc[:, cluster_labels]
        .values.T,
        cmap=cluster_cmap,
        norm=color_norm_clust,
    )

    clust_ax.set_yticks([])
    clust_ax.set_xticks([])
    clust_ax.set_ylim(len(cur_sample_labels) + 0.5, 0.5)
    clust_ax.set_title("Clusters", fontsize=16)

    # Plot the copy number heatmaps
    for ax, allele in zip(cn_axes, alleles):
        heatmap_data = (
            input_df.loc[cur_sample_labels]
            .astype(int)
            .unstack("sample_id")
            .loc[:, (allele)]
            .loc[:, cur_sample_labels]
            .values.T
        )

        im = ax.pcolormesh(
            x_pos,
            y_pos,
            heatmap_data,
            cmap=cmap,
            norm=color_norm,
            shading="auto",
        )

        # Plot chromosome boundaries
        for line in chr_ends.values:
            ax.axvline(x_pos[line], color="black", linewidth=0.75)

        # Set x-ticks and labels
        xtick_pos = (
            np.append([0], x_pos[chr_ends.values][:-1])
            + np.roll(np.append([0], x_pos[chr_ends.values][:-1]), -1)
        ) / 2
        xtick_pos[-1] += x_pos[-1] / 2
        ax.set_xticks(xtick_pos)
        ax.set_xticklabels(
            [x[3:] for x in chr_ends.index], ha="center", rotation=90, va="bottom"
        )
        ax.tick_params(width=0)
        ax.xaxis.set_tick_params(labelbottom=False, labeltop=True, bottom=False)
        ax.set_yticks([])

    # Plot the colorbar
    cax.pcolormesh(
        [0, 1],
        np.arange(0, cmax + 2),
        np.arange(0, cmax + 1)[:, np.newaxis],
        cmap=cmap,
        norm=color_norm,
    )

    cax.set_xticks([])
    cax.set_yticks(np.arange(0, cmax + 1) + 0.5)
    if np.max(input_df.values.astype(int)) > cmax:
        cax.set_yticklabels(
            [f"{x}+" if x == cmax else str(x) for x in np.arange(0, cmax + 1)],
            ha="left",
        )
    else:
        cax.set_yticklabels(np.arange(0, cmax + 1), ha="left")
    cax.yaxis.set_tick_params(left=False, labelleft=False, labelright=True)

    # Adjust y-limits for alignment
    for ax in [clust_ax] + list(cn_axes):
        ax.set_ylim(len(cur_sample_labels) + 0.5, 0.5)

    # Reduce space between subplots
    plt.subplots_adjust(wspace=0.05, hspace=0.05)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    return fig

def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights

In [ ]:
tree_path = "/home/ganesank/project/phytclust/Data/spectrum/SPECTRUM-OV-045__neither_final_tree.new"
tree = Phylo.read(tree_path, "newick", rooted=True)

clust_obj = PhytClust(tree, should_plot_scores=True, num_peaks=3, resolution_on=True, outgroup = "diploid")


get data

In [ ]:
clust_obj.best_peaks

In [ ]:
clust_obj_clusters = [
    clust_obj.clusters[clust_obj.best_peaks[0] - 1],
]
comparison_ids = ["6"]  # , "38", "3105"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])
df.set_index(["leaf_name", "comparison_IDs"], inplace=True)
# print(df)
# fig2 = plot_multiple_clusters(df, tree, outgroup="diploid")

input_file = "/home/ganesank/project/phytclust/Data/spectrum/SPECTRUM-OV-045__neither_final_cn_profiles.tsv"

cn_df = medicc.io.read_and_parse_input_data(
    input_file, total_copy_numbers=False, allele_columns=["cn_a", "cn_b"]
)
# fig = medicc.plot.plot_cn_heatmap(
#     cn_df, final_tree=tree, total_copy_numbers=False, alleles=["cn_a", "cn_b"], tree_marker_size = 50
# )

plot

In [ ]:
%matplotlib inline
fig = plot_cn_heatmap(
    cn_df,
    df,
    final_tree=tree,
    total_copy_numbers=False,
    alleles=["cn_a", "cn_b"],
    tree_marker_size=1,
    save_path = "/home/ganesank/project/phytclust/figs/fig.png",

)

In [ ]:
newick_str

In [ ]:
import dendropy
import io

newick_file = (
    "/home/ganesank/project/phytclust/Data/HIV_Trees_NG_FH/164Z_20241206.newick"
)

# Create a TreeList object to hold multiple trees
tree_list = dendropy.TreeList.get(
    path=newick_file,
    schema="newick",
    preserve_underscores=True,  # Adjust based on your data
)

# Check the number of trees loaded
print(f"Number of trees loaded: {len(tree_list)}")

consensus_tree = tree_list.consensus(min_freq=0.5)
newick_str = consensus_tree.as_string(schema="newick")

tree = Phylo.read(io.StringIO(newick_str), "newick", rooted=True)

In [ ]:
clust_obj = PhytClust(
    tree, should_plot_scores=True, num_peaks=3, resolution_on=True
)

In [ ]:
for clades in tree.find_clades():
    print(clades.branch_length)

Now, we calculate median CNP per cluster, IQR and plot the consensus CNP per MRCA with deviation

In [ ]:
import pandas as pd
from Bio.Phylo.BaseTree import Clade

clade_dict = clust_obj.clusters[clust_obj.best_peaks[0] - 1]

df = pd.read_csv(
    "/home/ganesank/project/phytclust/Data/spectrum/SPECTRUM-OV-003__neither_final_cn_profiles.tsv",
    sep="\t",
)

clade_names = [clade.name for clade in clade_dict.keys()]
cluster_numbers = [clade_dict[clade] for clade in clade_dict.keys()]

clade_df = pd.DataFrame({"clade_name": clade_names, "cluster_number": cluster_numbers})

merged_df = df.merge(clade_df, left_on="sample_id", right_on="clade_name", how="inner")

grouped = merged_df.groupby(["chrom", "start", "end", "cluster_number"])
median_values = grouped[["cn_a", "cn_b"]].median().reset_index()

def calculate_iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

iqr_values = grouped[["cn_a", "cn_b"]].apply(calculate_iqr).reset_index()
result = median_values.merge(
    iqr_values,
    on=["chrom", "start", "end", "cluster_number"],
    suffixes=("_median", "_iqr"),
)


# prune tree till MRCAs

In [ ]:
from Bio import Phylo
from Bio.Phylo.BaseTree import Clade
from copy import deepcopy

tree = deepcopy(clust_obj.tree)
clades_by_cluster = {}
for clade, cluster_number in clade_dict.items():
    if cluster_number not in clades_by_cluster:
        clades_by_cluster[cluster_number] = []
    clades_by_cluster[cluster_number].append(clade.name)

mrcas = {}
for cluster_number, clades in clades_by_cluster.items():
    if len(clades) == 1:
        mrcas[cluster_number] = clades[0]
    else:
        mrcas[cluster_number] = tree.common_ancestor(clades).name

nodes_to_keep = set(mrcas.values())

def recursive_collapse(clade):
    """Recursively collapse all descendants of a given clade."""
    for child in clade.clades[:]:  # Copy list of children to safely iterate
        if child.clades:  # If the child has its own descendants
            recursive_collapse(child)  # Collapse its subtree first
        clade.collapse(child)  # Collapse the child itself


def collapse_descendants(tree, nodes_to_keep):
    """Collapses all descendants of specified nodes, keeping the nodes themselves intact."""
    for clade in tree.find_clades():
        if clade.name in nodes_to_keep:
            recursive_collapse(clade)  # Collapse entire subtree of this clade


# Run the collapse function
collapse_descendants(tree, nodes_to_keep)

In [ ]:
df = result
cluster_mapping = mrcas
df.rename(columns={"cn_a_median": "cn_a", "cn_b_median": "cn_b"}, inplace=True)
df.drop(columns=["cn_a_iqr", "cn_b_iqr"], inplace=True)
df["cluster_number"] = df["cluster_number"].map(cluster_mapping)
df.rename(columns={"cluster_number": "sample_id"}, inplace=True)
df.to_csv(
    "/home/ganesank/project/phytclust/notebooks/OV-003-trunc.tsv", index=False, sep="\t"
)
print(df)


In [ ]:
import pandas as pd

input_file = "/home/ganesank/project/phytclust/notebooks/OV-003-trunc.tsv"
import medicc

cn_df = medicc.io.read_and_parse_input_data(
    input_file, total_copy_numbers=False, allele_columns=["cn_a", "cn_b"]
)
fig = medicc.plot.plot_cn_heatmap(
    cn_df, final_tree=tree, total_copy_numbers=False, alleles=["cn_a", "cn_b"], tree_marker_size = 50, tree_width_ratio = 0.3, tree_line_width = 1
)
%matplotlib inline

In [ ]:
result.groupby("cluster_number")["cn_a_iqr"].apply(lambda x: x.unique().mean())

In [ ]:
fig = medicc.plot.plot_cn_profiles(
    cn_df,
    input_tree=tree,
    # total_copy_numbers=False,
    allele_columns=["cn_a", "cn_b"],
    # tree_marker_size=50,
)

In [ ]:
import os
import io
import dendropy
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
from Bio import Phylo
import matplotlib.pyplot as plt
import pandas as pd


def generate_consensus_csv(cluster_dicts, output_csv_path):
    """
    Generates a CSV file from a list of cluster dictionaries.

    Parameters:
    - cluster_dicts (list of dict): Each dict maps clade names to integer labels.
    - output_csv_path (str): Path to save the generated CSV.
    """
    # Extract all unique leaf names
    leaf_names = set()
    for cluster in cluster_dicts:
        leaf_names.update(cluster.keys())
    leaf_names = sorted(leaf_names)

    # Initialize DataFrame with Leaf names
    df = pd.DataFrame({"Leaf": leaf_names})

    # Process each cluster dict
    for idx, cluster in enumerate(cluster_dicts, start=1):
        # Ensure cluster is a dictionary
        if not isinstance(cluster, dict):
            print(f"Warning: Cluster at index {idx} is not a dictionary. Skipping.")
            continue

        # Find the maximum label in the current cluster
        if cluster.values():
            max_label = max(cluster.values())
        else:
            max_label = 0  # Default if cluster is empty

        column_name = str(max_label + 1)

        # Assign label +1 to each leaf
        cluster_labels = {leaf: (label + 1) for leaf, label in cluster.items()}

        # Create a list of labels for all leaves
        labels = [cluster_labels.get(leaf, 0) for leaf in leaf_names]

        # Add the cluster assignment to the DataFrame
        df[column_name] = labels

    # Save the DataFrame to a CSV file
    df.to_csv(output_csv_path, index=False)
    print(f"CSV saved to {output_csv_path}")


def generate_consensus_tree(
    input_newick_path, output_newick_path, output_csv_path, min_freq=0.5
):
    """
    Generates a consensus tree from multiple trees in a Newick file and saves it along with a CSV of cluster assignments.

    Parameters:
    - input_newick_path (str): Path to the input Newick file containing multiple trees.
    - output_newick_path (str): Path to save the consensus Newick tree.
    - output_csv_path (str): Path to save the consensus CSV file.
    - min_freq (float): Minimum frequency (0 < min_freq <=1) for a clade to be included in the consensus tree.
    """
    try:
        # Load multiple trees into a TreeList object using DendroPy
        tree_list = dendropy.TreeList.get(
            path=input_newick_path, schema="newick", preserve_underscores=True
        )

        num_trees = len(tree_list)
        print(f"Loaded {num_trees} trees from {input_newick_path}")

        if num_trees == 0:
            print(f"No trees found in {input_newick_path}. Skipping file.")
            return

        # Generate a Majority-Rule Consensus Tree
        consensus_tree = tree_list.consensus(min_freq=min_freq)

        # Check if consensus tree has any clades
        if consensus_tree is None or len(consensus_tree.leaf_nodes()) == 0:
            print(
                f"Consensus tree for {input_newick_path} is empty or unresolved. Skipping file."
            )
            return

        # Optional: Ladderize the tree for better visualization
        consensus_tree.ladderize()

        # Export consensus tree to Newick string
        newick_str = consensus_tree.as_string(schema="newick").strip()

        # Convert Newick string to Bio.Phylo Tree object
        newick_io = io.StringIO(newick_str)
        phylo_tree = Phylo.read(newick_io, "newick")
        Phylo.draw(phylo_tree)
        clust_obj = PhytClust(
            phylo_tree,
            should_plot_scores=True,
            num_peaks=3,
            resolution_on=True,
            outgroup="HIV-1 HXB2 - env gene",
        )
        print(clust_obj.best_peaks)

        # Verify that clust_obj.clusters is not None and is a list
        if clust_obj.clusters is None:
            print(
                f"PhytClust returned None for clusters in {input_newick_path}. Skipping CSV generation."
            )
            return

        if not isinstance(clust_obj.clusters, list):
            print(
                f"PhytClust clusters are not in list format in {input_newick_path}. Skipping CSV generation."
            )
            return

        # Optional: Print cluster information for debugging
        print(f"Number of cluster assignments: {len(clust_obj.clusters)}")
        for idx, cluster in enumerate(clust_obj.clusters, start=1):
            print(f"Cluster {idx}: {cluster}")

        # Generate CSV from cluster assignments
        generate_consensus_csv(clust_obj.clusters, output_csv_path)

        # Save the consensus tree to a Newick file
        with open(output_newick_path, "w") as f_out_newick:
            f_out_newick.write(newick_str + "\n")  # Ensure newline at the end
        print(f"Consensus tree saved to {output_newick_path}")


    except Exception as e:
        print(f"An error occurred while processing {input_newick_path}: {e}")


def main():
    # Define the input directory containing .newick files
    input_directory = "/home/ganesank/project/phytclust/Data/HIV_Trees_NG_FH"

    # Define the output directory to save consensus trees and outputs
    output_directory = os.path.join(input_directory, "output")
    os.makedirs(output_directory, exist_ok=True)

    # Iterate over each file in the input directory
    for filename in os.listdir(input_directory):
        if filename.endswith(".newick"):
            input_newick_path = os.path.join(input_directory, filename)

            # Define output file paths
            base_filename = os.path.splitext(filename)[0]
            output_newick_filename = f"{base_filename}_consensus.newick"
            output_newick_path = os.path.join(output_directory, output_newick_filename)
            output_csv_filename = f"{base_filename}_clusters.csv"
            output_csv_path = os.path.join(output_directory, output_csv_filename)

            print(f"Processing file: {filename}")

            # Generate consensus tree and CSV
            generate_consensus_tree(
                input_newick_path=input_newick_path,
                output_newick_path=output_newick_path,
                output_csv_path=output_csv_path,
                min_freq=0.5,
            )


if __name__ == "__main__":
    main()